# RAG-Based Academic Research Search Engine

This notebook implements the full requirement from the hands-on question:

1. Load research papers from PDF files.
2. Preprocess and split the text.
3. Create structured documents using title, abstract, keywords and full text.
4. Generate Gemini embeddings.
5. Store embeddings in FAISS.
6. Retrieve papers using semantic similarity.
7. Combine semantic similarity with publication-year recency.
8. Return exactly `k` unique papers when at least `k` papers are available.
9. Display the selected papers in most-recent-first order.
10. Provide an interactive search.

**Gemini is used. OpenAI is not used.**

Put your PDFs in a folder named `papers` next to this notebook.


## 1. Install packages

Run this once in the VS Code terminal:

```bash
python -m pip install google-genai python-dotenv pandas numpy faiss-cpu pypdf
```

`os` is already part of Python. You do **not** install it with pip.


In [1]:
# 1. Import packages

import os
import re
import numpy as np
import pandas as pd
import faiss

from pathlib import Path
from dotenv import load_dotenv
from pypdf import PdfReader

from google import genai
from google.genai import types

print("Packages loaded successfully")


Packages loaded successfully


## 2. Load Gemini API key

Create `.env` in the same folder as this notebook:

```text
GEMINI_API_KEY=your_api_key_here
```

Never paste the real API key directly into the notebook.


In [2]:
# 2. Load Gemini API key

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY was not found in the .env file.")

client = genai.Client(api_key=api_key)

EMBEDDING_MODEL = "gemini-embedding-001"

print("Gemini client is ready")


Gemini client is ready


## 3. Find the research papers

Create this structure:

```text
GENAI_TUSHAR/
│
├── solution.ipynb
├── .env
│
└── papers/
    ├── AI_Research_paper.pdf
    ├── paper2.pdf
    ├── paper3.pdf
    ├── paper4.pdf
    └── paper5.pdf
```

Three papers are enough for an initial test. Five papers are better for demonstrating ranking and recency.


In [3]:
# 3. Paper folder

PAPER_FOLDER = Path("papers")
PAPER_FOLDER.mkdir(exist_ok=True)

pdf_files = sorted(PAPER_FOLDER.glob("*.pdf"))

print(f"PDF files found: {len(pdf_files)}")

for file in pdf_files:
    print("-", file.name)

if not pdf_files:
    raise FileNotFoundError(
        "No PDF files found. Put your research papers inside the 'papers' folder."
    )


PDF files found: 6
- AI_Research_paper.pdf
- paper2.pdf
- paper3.pdf
- paper4.pdf
- paper5.pdf
- paper6.pdf


## 4. Paper metadata

The assessment describes structured information such as title, abstract, full text, keywords and publication year.

PDF extraction can get the full text, but publication year and clean metadata are not always reliable. Therefore we keep the important metadata explicitly.

**Change the filenames below so they exactly match the PDFs you actually use.**

The first row matches the PDF you showed earlier.


In [4]:
# 4. Metadata for the papers

metadata = {
    "AI_Research_paper.pdf": {
        "title": "Research Paper on Artificial Intelligence & Its Implications",
        "year": 2023,
        "keywords": "artificial intelligence, machine learning, intelligent machines"
    },

    "paper2.pdf": {
        "title": "Public-Key Encryption with Quantum Keys",
        "year": 2023,
        "keywords": "quantum cryptography, public key encryption, quantum keys"
    },

    "paper3.pdf": {
        "title": "Post-Quantum Cryptography",
        "year": 2024,
        "keywords": "post quantum cryptography, cryptography, quantum computing"
    },

    "paper4.pdf": {
        "title": "Post Quantum Cryptography and its Comparison with Classical Cryptography",
        "year": 2024,
        "keywords": "post quantum cryptography, classical cryptography, quantum security"
    },

    "paper5.pdf": {
        "title": "Learning with Errors is easy with quantum samples",
        "year": 2018,
        "keywords": "learning with errors, LWE, quantum samples, post quantum cryptography"
    }
}

print("Metadata entries:", len(metadata))


Metadata entries: 5


## 5. Extract text from each PDF

Each PDF is converted to plain text.


In [5]:
# 5. Extract PDF text

def extract_pdf_text(file_path):
    reader = PdfReader(str(file_path))
    pages = []

    for page in reader.pages:
        text = page.extract_text() or ""
        pages.append(text)

    return "\n".join(pages)


papers = []

for pdf_file in pdf_files:
    text = extract_pdf_text(pdf_file)

    info = metadata.get(
        pdf_file.name,
        {
            "title": pdf_file.stem,
            "year": 2024,
            "keywords": ""
        }
    )

    papers.append({
        "file": pdf_file.name,
        "title": info["title"],
        "year": int(info["year"]),
        "keywords": info["keywords"],
        "full_text": text
    })

papers_df = pd.DataFrame(papers)

print(papers_df[["file", "title", "year"]])


                    file                                              title  \
0  AI_Research_paper.pdf  Research Paper on Artificial Intelligence & It...   
1             paper2.pdf            Public-Key Encryption with Quantum Keys   
2             paper3.pdf                          Post-Quantum Cryptography   
3             paper4.pdf  Post Quantum Cryptography and its Comparison w...   
4             paper5.pdf  Learning with Errors is easy with quantum samples   
5             paper6.pdf                                             paper6   

   year  
0  2023  
1  2023  
2  2024  
3  2024  
4  2018  
5  2024  


## 6. Preprocess the text

We clean repeated whitespace and obvious PDF extraction noise without aggressively removing technical terms.


In [6]:
# 6. Preprocess text

def clean_text(text):
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


papers_df["full_text"] = papers_df["full_text"].apply(clean_text)
papers_df["word_count"] = papers_df["full_text"].str.split().str.len()

print(papers_df[["title", "year", "word_count"]])


                                               title  year  word_count
0  Research Paper on Artificial Intelligence & It...  2023        2504
1            Public-Key Encryption with Quantum Keys  2023        7942
2                          Post-Quantum Cryptography  2024       13458
3  Post Quantum Cryptography and its Comparison w...  2024        8658
4  Learning with Errors is easy with quantum samples  2018        4049
5                                             paper6  2024        3334


## 7. Extract a simple abstract

If an `ABSTRACT` section can be found, it is used. Otherwise the beginning of the paper is used as a fallback.


In [7]:
# 7. Extract abstract

def extract_abstract(text):
    match = re.search(
        r"\bABSTRACT\b\s*(.*?)(?=\bINTRODUCTION\b|\b1\.?\s+INTRODUCTION\b)",
        text,
        flags=re.IGNORECASE
    )

    if match:
        abstract = match.group(1).strip()
        if len(abstract) > 100:
            return abstract[:5000]

    return text[:5000]


papers_df["abstract"] = papers_df["full_text"].apply(extract_abstract)

print(papers_df[["title", "abstract"]].head(1).to_string(index=False))


                                                       title                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           abstract
Research Paper on Artificial Intelligence & Its Implications - It is the science and engineering of making intelligent machines, especia

## 8. Create structured research-paper documents

The embedding input contains:

- title
- publication year
- keywords
- abstract
- full text

This directly addresses the structured-document requirement.


In [8]:
# 8. Create structured documents

def make_document(row):
    return f"""
Title: {row['title']}
Publication Year: {row['year']}
Keywords: {row['keywords']}

Abstract:
{row['abstract']}

Full Text:
{row['full_text']}
""".strip()


papers_df["document"] = papers_df.apply(make_document, axis=1)

print(papers_df["document"].iloc[0][:1500])


Title: Research Paper on Artificial Intelligence & Its Implications
Publication Year: 2023
Keywords: artificial intelligence, machine learning, intelligent machines

Abstract:
- It is the science and engineering of making intelligent machines, especially intelligent computer programs. It is related to the similar task of using computers to understand human intelligence, but AI does not have to confine itself to methods that are biologically observable. While no consensual definition of Artificial Intelligence (AI) exists, AI is broadly characteriz ed as the study of computa tions that allow for perception, reason and action. Today, the amount of data that is generated, by both humans and machines, far outpaces humans’ ability to absorb, interpret, and make complex decisions based on that data. Artificial intelligence forms th e basis for all computer learning and is the future of all complex decision making. This paper examines features of artificial Intelligence,

Full Text:
© 2023 IJ

## 9. Split long documents into chunks

Long papers are divided into overlapping chunks so that embedding requests stay manageable.

The title, year and keywords remain in every structured document before splitting.


In [9]:
# 9. Split documents into chunks

def split_text(text, chunk_size=3500, overlap=400):
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])

        if end == len(text):
            break

        start = end - overlap

    return chunks


chunks = []

for _, row in papers_df.iterrows():
    text_chunks = split_text(row["document"])

    for chunk_number, chunk in enumerate(text_chunks):
        chunks.append({
            "file": row["file"],
            "title": row["title"],
            "year": row["year"],
            "keywords": row["keywords"],
            "chunk_number": chunk_number,
            "text": chunk
        })

chunks_df = pd.DataFrame(chunks)

print("Total chunks:", len(chunks_df))
print(chunks_df[["title", "year", "chunk_number"]].head())


Total chunks: 86
                                               title  year  chunk_number
0  Research Paper on Artificial Intelligence & It...  2023             0
1  Research Paper on Artificial Intelligence & It...  2023             1
2  Research Paper on Artificial Intelligence & It...  2023             2
3  Research Paper on Artificial Intelligence & It...  2023             3
4  Research Paper on Artificial Intelligence & It...  2023             4


## 10. Generate Gemini document embeddings

For retrieval, document embeddings use the `RETRIEVAL_DOCUMENT` task.

The current Google GenAI SDK uses `contents=...`.

This is important because the older code used an incorrect `content=` argument, which caused the error you saw.


In [18]:
# 10. Generate embeddings

texts = [str(chunk) for chunk in chunks]

result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=texts,
    config=types.EmbedContentConfig(
        task_type="RETRIEVAL_DOCUMENT",
        output_dimensionality=768
    )
)

embeddings = np.array(
    [item.values for item in result.embeddings],
    dtype="float32"
)

print("Embedding created")
print("Shape:", embeddings.shape)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

## 11. Create the FAISS index

We normalize the vectors and use an inner-product index, which gives cosine-style similarity for normalized vectors.


In [11]:
# 11. Create FAISS index

faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("FAISS index created")
print("Vectors stored:", index.ntotal)
print("Vector dimension:", dimension)


AttributeError: 'list' object has no attribute 'shape'

## 12. Create the query embedding

Queries use the `RETRIEVAL_QUERY` task.


In [ ]:
# 12. Query embedding

def get_query_embedding(question):
    result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=question,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY",
            output_dimensionality=768
        )
    )

    vector = np.asarray(
        result.embeddings[0].values,
        dtype="float32"
    ).reshape(1, -1)

    faiss.normalize_L2(vector)

    return vector


print("Query embedding function is ready")


## 13. Semantic candidate search

FAISS retrieves more candidates than the final `k`.

This is intentional: the additional challenge says relevance depends on both semantic similarity and publication recency.


In [ ]:
# 13. Semantic candidate search

def semantic_candidates(question, candidate_count=20):
    query_vector = get_query_embedding(question)

    candidate_count = min(candidate_count, index.ntotal)

    scores, positions = index.search(query_vector, candidate_count)

    rows = []

    for score, position in zip(scores[0], positions[0]):
        if position == -1:
            continue

        row = chunks_df.iloc[int(position)].copy()
        row["semantic_score"] = float(score)

        rows.append(row)

    return pd.DataFrame(rows)


print("Semantic search function is ready")


## 14. Apply the additional challenge: semantic similarity + recency

The assessment says document relevance depends on:

1. semantic similarity to the query
2. recency of the document

We therefore calculate both.

Weights used here:

- 70% semantic similarity
- 30% recency

The final selected papers are then displayed in **most-recent-first order**, matching the stated output requirement.


In [ ]:
# 14. Semantic + recency ranking

def rank_results(candidates, k=3):
    if k <= 0:
        raise ValueError("k must be greater than 0.")

    result = candidates.copy()

    # Keep only the best chunk for each paper.
    result = (
        result
        .sort_values("semantic_score", ascending=False)
        .drop_duplicates("file")
        .copy()
    )

    if len(result) < k:
        raise ValueError(
            f"Only {len(result)} unique papers are available, "
            f"but exactly {k} papers were requested."
        )

    # Normalize semantic similarity.
    min_score = result["semantic_score"].min()
    max_score = result["semantic_score"].max()

    if max_score == min_score:
        result["semantic_norm"] = 1.0
    else:
        result["semantic_norm"] = (
            (result["semantic_score"] - min_score)
            / (max_score - min_score)
        )

    # Normalize publication year.
    min_year = result["year"].min()
    max_year = result["year"].max()

    if max_year == min_year:
        result["recency_score"] = 1.0
    else:
        result["recency_score"] = (
            (result["year"] - min_year)
            / (max_year - min_year)
        )

    # Combined relevance score.
    result["final_score"] = (
        0.70 * result["semantic_norm"]
        + 0.30 * result["recency_score"]
    )

    # Select exactly k using both factors.
    result = (
        result
        .sort_values(
            ["final_score", "semantic_score"],
            ascending=[False, False]
        )
        .head(k)
    )

    # Final output order required by the problem statement.
    result = result.sort_values(
        ["year", "final_score"],
        ascending=[False, False]
    )

    return result.reset_index(drop=True)


print("Ranking function is ready")


## 15. Complete search function

Flow:

**Question → Gemini query embedding → FAISS semantic retrieval → semantic + recency ranking → exactly k unique papers**


In [ ]:
# 15. Complete search function

def search(question, k=3):
    if not question.strip():
        raise ValueError("Question cannot be empty.")

    if k > len(papers_df):
        raise ValueError(
            f"You requested {k} papers, but only {len(papers_df)} "
            "unique research papers are available."
        )

    candidates = semantic_candidates(
        question,
        candidate_count=min(len(chunks_df), max(20, k * 4))
    )

    return rank_results(candidates, k=k)[
        [
            "title",
            "year",
            "keywords",
            "semantic_score",
            "recency_score",
            "final_score"
        ]
    ]


print("Search engine is ready")


## 16. Test the engine

Try the scenario from the assessment:

`quantum computing and cryptography`


In [ ]:
# 16. Test search

question = "quantum computing and cryptography"

results = search(question, k=3)

display(results)


## 17. Human-readable results


In [ ]:
# 17. Print results

def print_results(question, k=3):
    results = search(question, k)

    print("\nQuery:", question)
    print("=" * 70)

    for i, row in results.iterrows():
        print(f"Result {i + 1}")
        print("Title :", row["title"])
        print("Year  :", row["year"])
        print("Score :", round(row["final_score"], 4))
        print("Keywords:", row["keywords"])
        print("-" * 70)


print_results("quantum computing and cryptography", k=3)


## 18. Interactive search

The user can enter a query and request the exact number of papers.

Type `exit` to stop.


In [ ]:
# 18. Interactive search

while True:
    question = input("\nEnter your research question (or type 'exit'): ")

    if question.lower().strip() == "exit":
        print("Search ended.")
        break

    try:
        k = int(input("How many papers do you want? "))
        print_results(question, k)

    except Exception as error:
        print("Error:", error)


# Final requirement checklist

| Requirement | Implemented |
|---|---|
| Load research papers | Yes |
| Preprocess papers | Yes |
| Tokenize/split long text | Yes |
| Generate structured documents | Yes |
| Title | Yes |
| Abstract | Yes |
| Full text | Yes |
| Keywords | Yes |
| Publication year | Yes |
| Gemini embeddings | Yes |
| FAISS index | Yes |
| Semantic search | Yes |
| Top-k retrieval | Yes |
| Semantic similarity + recency | Yes |
| Exactly k unique papers | Yes, when at least k exist |
| Most-recent-first output | Yes |
| Interactive query | Yes |
| OpenAI | No |

**For the actual hands-on, if the platform supplies a dataset, use that supplied dataset instead of the example papers.**
